### ЗАДАЧА: Ежедневная выгрузка возвратов маркетплейса

Команда маркетплейса в конце дня получает журнал возвратов в виде строк.
На его основе нужно подготовить два разных результата:
- управленческий отчет в `JSON` для аналитиков и менеджеров,
- полный внутренний снимок в `pickle` для разборов, аудита и восстановления состояния.

НЕОБХОДИМО:

1. Преобразовать строки из `rows` в список словарей `returns_data`.
2. Для каждого возврата привести типы:
   - `refund_amount` -> `float`
   - `is_fraud_flag` -> `bool`
   - `items` -> `list[dict]`, где каждый элемент имеет поля `sku` и `qty`
3. Для каждого возврата вычислить поле `total_units`.
4. Построить словарь `seller_summary`, где по каждому продавцу будет:
   - `returns_count`
   - `approved_refund_amount`
   - `fraud_flags`
   - `total_units`
5. Построить словарь `warehouse_summary`, где по каждому складу будет:
   - `returns_count`
   - `approved_refund_amount`
6. Собрать список `escalations` по возвратам, которые требуют ручной проверки.
   Возврат нужно добавить в `escalations`, если:
   - `is_fraud_flag == True`,
   - или `refund_amount >= 15000`,
   - или `status == 'manual_review'`.
7. Для каждого элемента в `escalations` оставить только поля:
   - `return_id`
   - `seller`
   - `warehouse`
   - `reason`
   - `refund_amount`
   - `status`
8. Отсортировать `escalations` по `refund_amount` по убыванию.
9. Собрать `public_report` со структурой:
   - `report_name`
   - `returns_count`
   - `seller_summary`
   - `warehouse_summary`
   - `escalations`
10. Сохранить `public_report` в файл `returns_report.json`.
11. Сохранить полный снимок в файл `returns_snapshot.pkl`.
12. Прочитать оба файла обратно и вывести результаты.


In [ ]:
import json
import pickle


# row format: return_id|seller|warehouse|reason|items|refund_amount|is_fraud_flag|status
# items format: sku:qty;sku:qty;...
rows = [
    "R-301|TechStore|WH-MSK|damaged|KB-11:1;MS-02:2|7400.00|false|approved",
    "R-302|HomeHub|WH-SPB|wrong_item|LAM-77:1|15990.00|false|manual_review",
    "R-303|TechStore|WH-MSK|fraud_suspected|HD-55:3|22100.50|true|manual_review",
    "R-304|UrbanWear|WH-EKB|size_issue|TS-10:2;JN-02:1|4990.00|false|approved",
    "R-305|HomeHub|WH-SPB|damaged|CHR-22:1|8900.00|false|rejected",
    "R-306|UrbanWear|WH-EKB|fraud_suspected|SN-88:4|12400.00|true|approved",
    "R-307|TechStore|WH-KZN|wrong_item|MON-31:1|18990.00|false|approved",
]

returns_data = []
seller_summary = {}
warehouse_summary = {}
escalations = []

for row in rows:
    return_id, seller, warehouse, reason, items_raw, refund_amount_raw, is_fraud_flag_raw, status = row.split("|")

    # TODO 1: преобразуйте refund_amount_raw в float и сохраните в refund_amount
    # TODO 2: преобразуйте is_fraud_flag_raw в bool и сохраните в is_fraud_flag
    refund_amount = float(refund_amount_raw)
    is_fraud_flag = True if is_fraud_flag_raw == 'true' else False

    items = []
    total_units = 0

    for item_raw in items_raw.split(";"):
        sku, qty_raw = item_raw.split(":")

        # TODO 3: преобразуйте qty_raw в int и сохраните в qty
        qty = int(qty_raw)
        # TODO 4: добавьте в items словарь вида {"sku": ..., "qty": ...}
        items.append({"sku": sku, "qty": qty})
        # TODO 5: увеличьте total_units на qty
        total_units += qty

    return_record = {
        "return_id": return_id,
        "seller": seller,
        "warehouse": warehouse,
        "reason": reason,
        "items": items,
        # TODO 6: добавьте refund_amount
        'refund_amount':refund_amount,
        # TODO 7: добавьте is_fraud_flag
        'is_fraud_flag':is_fraud_flag,
        "status": status,
        # TODO 8: добавьте total_units
        'total_units':total_units
    }

    returns_data.append(return_record)

    # TODO 9: если seller еще отсутствует в seller_summary,
    # создайте для него словарь с полями:
    #   returns_count -> 0
    #   approved_refund_amount -> 0.0
    #   fraud_flags -> 0
    #   total_units -> 0
    if seller not in seller_summary:
        seller_summary[seller] = {
            'returns_count' :0,
            'approved_refund_amount' :0.0,
            'fraud_flags': 0,
            'total_units' : 0
        }


    # TODO 10: увеличьте seller_summary[seller]['returns_count'] на 1
    seller_summary[seller]['returns_count']+= 1
    # TODO 11: увеличьте seller_summary[seller]['total_units'] на total_units
    seller_summary[seller]['total_units']+= total_units
    # TODO 12: если is_fraud_flag == True, увеличьте fraud_flags на 1
    if is_fraud_flag == True:
        seller_summary[seller]['fraud_flags'] += 1
    # TODO 13: если status == 'approved', прибавьте refund_amount к approved_refund_amount
    if status == 'approved':
         seller_summary[seller]['approved_refund_amount'] += refund_amount

    # TODO 14: если warehouse еще отсутствует в warehouse_summary,
    # создайте словарь с полями:
    #   returns_count -> 0
    #   approved_refund_amount -> 0.0
    if warehouse not in warehouse_summary:
        warehouse_summary[warehouse] = {
            'returns_count':0,
            'approved_refund_amount':0.0
        }

    # TODO 15: увеличьте warehouse_summary[warehouse]['returns_count'] на 1
    warehouse_summary[warehouse]['returns_count'] += 1
    # TODO 16: если status == 'approved', прибавьте refund_amount к warehouse_summary[warehouse]['approved_refund_amount']
    if status == 'approved':
        warehouse_summary[warehouse]['approved_refund_amount'] += refund_amount
    # TODO 17: если возврат требует ручной проверки,
    # соберите словарь только с нужными полями и добавьте его в escalations
    if return_record['is_fraud_flag'] == True or refund_amount >= 15000 or status == 'manual_review':
        escalations.append({
            'return_id':return_id,
            'seller':seller,
            'warehouse':warehouse,
            'reason':reason,
            'refund_amount':refund_amount,
            'status':status
        })

# TODO 18: отсортируйте список escalations по ключу refund_amount по убыванию
escalations.sort(key=lambda x: x['refund_amount'], reverse=True)

# TODO 19: округлите все approved_refund_amount в seller_summary до 2 знаков
round(seller_summary[seller]['approved_refund_amount'],2)
# TODO 20: округлите все approved_refund_amount в warehouse_summary до 2 знаков
round(warehouse_summary[warehouse]['approved_refund_amount'],2)

public_report = {
    "report_name": "daily_marketplace_returns",
    "returns_count": len(returns_data),
    # TODO 21: добавьте seller_summary
    'seller_summary':seller_summary,
    # TODO 22: добавьте warehouse_summary
    'warehouse_summary':warehouse_summary,
    # TODO 23: добавьте escalations
    'escalations':escalations
}

snapshot = {
    "rows": rows,
    "returns_data": returns_data,
    "seller_summary": seller_summary,
    "warehouse_summary": warehouse_summary,
    "escalations": escalations,
}

# TODO 24: сохраните public_report в returns_report.json через json.dump
#   используйте ensure_ascii=False и indent=2
with open('returns_report.json','w') as f:
    json.dump(public_report,f,ensure_ascii=False,indent=2)

# TODO 25: сохраните snapshot в returns_snapshot.pkl через pickle.dump
with open('returns_snapshot.pkl','wb') as f:
    pickle.dump(snapshot,f)

# TODO 26: прочитайте returns_report.json в loaded_report через json.load
with open('returns_report.json','r') as f:
    loaded_report = json.load(f)
# TODO 27: прочитайте returns_snapshot.pkl в loaded_snapshot через pickle.load
with open('returns_snapshot.pkl','rb')as f:
    loaded_snapshot = pickle.load(f)

print("Полный список возвратов:")
print(returns_data)
print()

print("Управленческий отчет:")
print(public_report)
print()

print("Данные из JSON:")
print(loaded_report)
print()

print("Данные из pickle:")
print(loaded_snapshot)
print()

print("Возвраты на ручную проверку:")
print(escalations)


Полный список возвратов:
[{'return_id': 'R-301', 'seller': 'TechStore', 'warehouse': 'WH-MSK', 'reason': 'damaged', 'items': [{'sku': 'KB-11', 'qty': 1}, {'sku': 'MS-02', 'qty': 2}], 'refund_amount': 7400.0, 'is_fraud_flag': False, 'status': 'approved', 'total_units': 3}, {'return_id': 'R-302', 'seller': 'HomeHub', 'warehouse': 'WH-SPB', 'reason': 'wrong_item', 'items': [{'sku': 'LAM-77', 'qty': 1}], 'refund_amount': 15990.0, 'is_fraud_flag': False, 'status': 'manual_review', 'total_units': 1}, {'return_id': 'R-303', 'seller': 'TechStore', 'warehouse': 'WH-MSK', 'reason': 'fraud_suspected', 'items': [{'sku': 'HD-55', 'qty': 3}], 'refund_amount': 22100.5, 'is_fraud_flag': True, 'status': 'manual_review', 'total_units': 3}, {'return_id': 'R-304', 'seller': 'UrbanWear', 'warehouse': 'WH-EKB', 'reason': 'size_issue', 'items': [{'sku': 'TS-10', 'qty': 2}, {'sku': 'JN-02', 'qty': 1}], 'refund_amount': 4990.0, 'is_fraud_flag': False, 'status': 'approved', 'total_units': 3}, {'return_id': 'R-3